In [1]:
# ==== SETUP: Download data files ====
# Run this cell ONCE to download the data folder from Google Drive.
# After it finishes, you can skip this cell in future runs.

import subprocess, sys, os
subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown", "-q"])
import gdown
GOOGLE_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1ei8-m-N2vJHj2y7LHA7QYEC7FT2G-qDN?usp=sharing"

output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(GOOGLE_DRIVE_FOLDER_URL, output=output_dir, quiet=False)

print("Done! All data files downloaded to the 'data/' folder.")


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Retrieving folder contents


Processing file 1bCRsZL-M7X6i-oFI0OtUsrHv2Ehb6hTG .DS_Store
Processing file 1aeao9JFIaXAp2KIgUGfpFqqu1JGCDFjw crsp.csv
Processing file 18mL13MY4s9IPK6B3flAzKyuPh-Nw30AA F-F_Momentum_Factor.csv
Processing file 1F1S7Q7udEU208xfnDVbya3PQmbKq312D F-F_Research_Data_Factors.csv
Processing file 1vxD4-VAslFtBK_iGc7umDpt04atRVXbg ret.csv
Processing file 1YAgxHYNHBz3YDoTfOmBJaxtHJAKQQydO SIC_49_Industry.xlsx
Processing file 1dVVPlLlo5C-LnjtoU-UEaXcrpz3yPOOW signed_predictors_dl_wide.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1bCRsZL-M7X6i-oFI0OtUsrHv2Ehb6hTG
To: /Users/nnagelia/Documents/backtestingProject/data/.DS_Store
100%|██████████| 6.15k/6.15k [00:00<00:00, 12.3MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1aeao9JFIaXAp2KIgUGfpFqqu1JGCDFjw
From (redirected): https://drive.google.com/uc?id=1aeao9JFIaXAp2KIgUGfpFqqu1JGCDFjw&confirm=t&uuid=ace06f90-7f56-4586-8eab-2d1baa98de57
To: /Users/nnagelia/Documents/backtestingProject/data/crsp.csv
100%|██████████| 193M/193M [00:04<00:00, 42.3MB/s] 
Downloading...
From: https://drive.google.com/uc?id=18mL13MY4s9IPK6B3flAzKyuPh-Nw30AA
To: /Users/nnagelia/Documents/backtestingProject/data/F-F_Momentum_Factor.csv
100%|██████████| 21.5k/21.5k [00:00<00:00, 1.05MB/s]
Downloading...
From: https://drive.google.com/uc?id=1F1S7Q7udEU208xfnDVbya3PQmbKq312D
To: /Users/nnagelia/Documents/backtesting

Done! All data files downloaded to the 'data/' folder.



Download completed


In [8]:
import pandas as pd
#1: data colleciton
# Load all 4 files
ff_factors = pd.read_csv("data/F-F_Research_Data_Factors.csv", skiprows=3)
ff_mom = pd.read_csv("data/F-F_Momentum_Factor.csv", skiprows=13)
sic_mapping = pd.read_excel("data/SIC_49_Industry.xlsx")
oap_factors = pd.read_csv("data/signed_predictors_dl_wide.csv")
crsp = pd.read_csv("data/crsp.csv")

# Print heads
print("=== FF 3 Factors ===")
print(ff_factors.head())
print(f"\nShape: {ff_factors.shape}")

print("\n=== FF Momentum ===")
print(ff_mom.head())
print(f"\nShape: {ff_mom.shape}")

print("\n=== SIC 49 Industry Mapping ===")
print(sic_mapping.head())
print(f"\nShape: {sic_mapping.shape}")

print("\n=== OAP Factors ===")
print(oap_factors.head())
print(f"\nShape: {oap_factors.shape}")

print("\n=== CRSP ===")
print(crsp.head())
print(f"\nShape: {crsp.shape}")

=== FF 3 Factors ===
  Unnamed: 0   Mkt-RF      SMB      HML       RF
0     192607     2.89    -2.55    -2.39     0.22
1     192608     2.64    -1.14     3.81     0.25
2     192609     0.38    -1.36     0.05     0.23
3     192610    -3.27    -0.14     0.82     0.32
4     192611     2.54    -0.11    -0.61     0.31

Shape: (1298, 5)

=== FF Momentum ===
  Unnamed: 0      Mom
0     192701     0.57
1     192702    -1.50
2     192703     3.52
3     192704     4.36
4     192705     2.78

Shape: (1293, 2)

=== SIC 49 Industry Mapping ===
   Industry  SIC_start  SIC_end Industry_name
0         1        100      199         Agric
1         1        200      299         Agric
2         1        700      799         Agric
3         1        910      919         Agric
4         1       2048     2048         Agric

Shape: (598, 4)

=== OAP Factors ===
   permno  yyyymm  AM  AOP  AbnormalAccruals  Accruals  AccrualsBM  Activism1  \
0   10000  198601 NaN  NaN               NaN       NaN         NaN  

/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_30304/4241882157.py:8: DtypeWarning: Columns (0: SICCD) have mixed types. Specify dtype option on import or set low_memory=False.
  crsp = pd.read_csv("data/crsp.csv")


In [9]:
#2: Data cleaning
import pandas as pd
import numpy as np

#config the 5 and 100 thresholds are given per assignemnt
DATA_DIR = "data"
OUTPUT_DIR = "cleaned_data"
START_YEAR = 1985
END_YEAR = 2023
MIN_PRICE = 5.0
MIN_MARKET_CAP = 100

#CRSP cleaning:
before_rows = len(crsp)
crsp.columns = crsp.columns.str.upper()
crsp['DATE'] = pd.to_datetime(crsp['DATE'])
crsp['YEAR'] = crsp['DATE'].dt.year
crsp['MONTH'] = crsp['DATE'].dt.month
crsp['YYYYMM'] = crsp['YEAR'] * 100 + crsp['MONTH']
crsp = crsp[crsp['SHRCD'].isin([10, 11])] 
crsp = crsp[crsp['EXCHCD'].isin([1, 2, 3])]
crsp['RET'] = pd.to_numeric(crsp['RET'], errors='coerce')
crsp = crsp[crsp['RET'].notna()]
crsp = crsp[crsp['RET'] > -1.0] 
crsp['PRC_ABS'] = crsp['PRC'].abs()
crsp['MARKET_CAP'] = crsp['PRC_ABS'] * crsp['SHROUT'] 
crsp = crsp[crsp['PRC_ABS'] >= MIN_PRICE] 
crsp = crsp[crsp['MARKET_CAP'] >= MIN_MARKET_CAP * 1000] 
crsp = crsp[(crsp['YEAR'] >= START_YEAR) & (crsp['YEAR'] <= END_YEAR)]
 
print(f"CRSP cleaned: {len(crsp):,} rows. (Removed {before_rows - len(crsp):,} rows)")


CRSP cleaned: 1,270,055 rows. (Removed 2,420,213 rows)


In [10]:
# Vectorized FF49 mapping: interval lookup instead of per-row Python loop
sic_mapping_sorted = sic_mapping.sort_values('SIC_start').reset_index(drop=True)
intervals = pd.IntervalIndex.from_arrays(
    sic_mapping_sorted['SIC_start'],
    sic_mapping_sorted['SIC_end'],
    closed='both',
)
industry_values = sic_mapping_sorted['Industry'].to_numpy()

sic_numeric = pd.to_numeric(crsp['SICCD'], errors='coerce')
idx = intervals.get_indexer(sic_numeric.fillna(-1).to_numpy())
crsp['FF49'] = np.where(idx >= 0, industry_values[idx], 49).astype('int16')

# Vectorized annual compounding via log1p/expm1 + groupby.sum (C-level, no Python apply)
log1p_ret = np.log1p(crsp['RET'].to_numpy())
annual_returns = (
    pd.DataFrame({
        'PERMNO': crsp['PERMNO'].to_numpy(),
        'YEAR': crsp['YEAR'].to_numpy(),
        'LOG1P_RET': log1p_ret,
    })
    .groupby(['PERMNO', 'YEAR'], sort=False, as_index=False)['LOG1P_RET']
    .sum()
)
annual_returns['RET_ANNUAL'] = np.expm1(annual_returns['LOG1P_RET'])
annual_returns = annual_returns[['PERMNO', 'YEAR', 'RET_ANNUAL']]

print(f"Annual returns: {len(annual_returns):,} stock-years")

oap_rows_before = len(oap_factors)
dec_crsp = crsp.loc[crsp['MONTH'] == 12, ['PERMNO', 'YEAR', 'PRC_ABS', 'MARKET_CAP', 'SICCD', 'FF49']].copy()
oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
oap_factors['MONTH'] = oap_factors['yyyymm'] % 100
oap_factors_dec = oap_factors[oap_factors['MONTH'] == 12].copy()
oap_factors_dec['RETURN_YEAR'] = oap_factors_dec['YEAR'] + 1
print(f"oap_factors December: {len(oap_factors_dec):,} rows. Before: {oap_rows_before:,} rows.")


Annual returns: 124,875 stock-years


/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_30304/3489364595.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_30304/3489364595.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  oap_factors['MONTH'] = oap_factors['yyyymm'] % 100


oap_factors December: 454,907 rows. Before: 5,416,424 rows.


In [11]:
oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
oap_factors['MONTH'] = oap_factors['yyyymm'] % 100
oap_factors_dec = oap_factors[oap_factors['MONTH'] == 12].copy()
oap_factors_dec['RETURN_YEAR'] = oap_factors_dec['YEAR'] + 1  # Dec T factors predict T+1 returns
 
print(f"OAP December: {len(oap_factors_dec):,} rows")

merged = pd.merge(
    oap_factors_dec,
    annual_returns,
    left_on=['permno', 'RETURN_YEAR'],
    right_on=['PERMNO', 'YEAR'],
    how='inner'
)
 
# Merge with December CRSP data for FF49
merged = pd.merge(
    merged,
    dec_crsp,
    left_on=['permno', 'YEAR_x'],
    right_on=['PERMNO', 'YEAR'],
    how='left'
)
merged.head()


OAP December: 454,907 rows


,permno,yyyymm,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,...,RETURN_YEAR,PERMNO_x,YEAR_y,RET_ANNUAL,PERMNO_y,YEAR,PRC_ABS,MARKET_CAP,SICCD,FF49
0,10001,201212,2.055205,0.013963,0.014669,0.033528,NaN,NaN,NaN,NaN,...,2013,10001,2013,-0.014836,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,201312,2.078684,NaN,-0.032669,0.018642,NaN,NaN,NaN,NaN,...,2014,10001,2014,0.225805,NaN,NaN,NaN,NaN,NaN,NaN
2,10001,201412,1.762830,0.042705,-0.005713,0.034970,NaN,NaN,NaN,NaN,...,2015,10001,2015,-0.065084,10001.0,2014.0,11.02,115577.76,4925.0,31.0
3,10001,201512,2.734447,NaN,0.001784,0.018868,NaN,NaN,NaN,NaN,...,2016,10001,2016,0.656020,NaN,NaN,NaN,NaN,NaN,NaN
4,10001,201612,1.497349,NaN,-0.039445,0.063455,NaN,NaN,NaN,NaN,...,2017,10001,2017,0.043988,10001.0,2016.0,12.55,132026.00,4925.0,31.0


In [13]:
ff_factors.columns = ['YYYYMM', 'MKT_RF', 'SMB', 'HML', 'RF']
ff_factors = ff_factors[ff_factors['YYYYMM'].astype(str).str.len() == 6]
ff_factors['YYYYMM'] = ff_factors['YYYYMM'].astype(int)
ff_factors[['MKT_RF', 'SMB', 'HML', 'RF']] = ff_factors[['MKT_RF', 'SMB', 'HML', 'RF']].apply(pd.to_numeric, errors='coerce') / 100
 
ff_mom.columns = ['YYYYMM', 'UMD']
ff_mom = ff_mom[ff_mom['YYYYMM'].astype(str).str.len() == 6]
ff_mom['YYYYMM'] = ff_mom['YYYYMM'].astype(int)
ff_mom['UMD'] = pd.to_numeric(ff_mom['UMD'], errors='coerce') / 100
 
ff_all = pd.merge(ff_factors, ff_mom, on='YYYYMM', how='inner')
ff_all.head()

,YYYYMM,MKT_RF,SMB,HML,RF,UMD
0,192701,-0.0005,-0.0032,0.0458,0.0025,0.0057
1,192702,0.0417,0.0007,0.0272,0.0026,-0.0150
2,192703,0.0014,-0.0177,-0.0238,0.0030,0.0352
3,192704,0.0047,0.0039,0.0065,0.0025,0.0436
4,192705,0.0545,0.0155,0.0480,0.0030,0.0278


In [14]:
id_cols = ['permno', 'yyyymm', 'YEAR', 'MONTH', 'RETURN_YEAR', 'PERMNO', 'YEAR_x', 'YEAR_y',
           'RET_ANNUAL', 'PRC_ABS', 'MARKET_CAP', 'SICCD', 'FF49']
factor_cols = [c for c in merged.columns if c not in id_cols]
 
missing_pct = (merged[factor_cols].isna().sum() / len(merged) * 100).sort_values()
print("Missing data percentage per factor:")
print(missing_pct)

Missing data percentage per factor:
PERMNO_x                 0.000000
DivInit                  0.032327
DivOmit                  0.032327
MaxRet                   0.213530
ExchSwitch               0.239051
                          ...    
AccrualsBM              95.106680
Recomm_ShortInterest    96.615851
Activism2               98.102903
ProbInformedTrading     99.029333
IO_ShortInterest        99.196073
Length: 211, dtype: float64


In [26]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
merged = merged[merged['FF49'].notna()].copy()
merged.to_csv(f"{OUTPUT_DIR}/cleaned_merged_data.csv", index=False)
ff_all.to_csv(f"{OUTPUT_DIR}/ff_factors_clean.csv", index=False)


In [27]:
print(merged.head())
print(ff_all.head())

    permno  yyyymm        AM       AOP  AbnormalAccruals  Accruals  \
2    10001  201412  1.762830  0.042705         -0.005713  0.034970   
4    10001  201612  1.497349       NaN         -0.039445  0.063455   
6    10002  199712  3.365251       NaN               NaN -0.010779   
7    10002  199812  3.135689       NaN               NaN  0.033185   
11   10002  200312  4.752564       NaN               NaN -0.011305   

    AccrualsBM  Activism1  Activism2     AdExp  ...  RETURN_YEAR  PERMNO_x  \
2          NaN        NaN        NaN       NaN  ...         2015     10001   
4          NaN        NaN        NaN       NaN  ...         2017     10001   
6          NaN        NaN        NaN  0.001250  ...         1998     10002   
7          NaN        NaN        NaN  0.001425  ...         1999     10002   
11         NaN        NaN        NaN  0.002777  ...         2004     10002   

    YEAR_y  RET_ANNUAL  PERMNO_y    YEAR  PRC_ABS  MARKET_CAP   SICCD  FF49  
2     2015   -0.065084   10001.0